In [1]:
from pathlib import Path

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
)
from conversational_toolkit.embeddings.qwen_vl import Qwen3VLEmbeddings
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import EMBEDDING_MODEL
from loguru import logger
from dotenv import load_dotenv

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


/Users/tloiseau/Documents/SDSC Projects/sme-kt-zh-collaboration-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv("../../.env.local")

_DB_DIR = Path("../src/sme_kt_zh_collaboration_rag/db")

IMAGE_VS_PATH = _DB_DIR / "vs_image"
TEXT_VS_PATH = _DB_DIR / "vs_text"

logger.info("Initializing vector stores...")
all_chunks = load_chunks()

2026-03-02 20:31:06.729 | INFO     | __main__:<module>:8 - Initializing vector stores...
2026-03-02 20:31:06.730 | WARNING  | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:201 - Skipping unsupported file type '': .DS_Store
2026-03-02 20:31:06.730 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:208 - Chunking 34 files from /Users/tloiseau/Documents/SDSC Projects/sme-kt-zh-collaboration-rag/data
2026-03-02 20:31:07.019 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:227 -   ART_internal_procurement_policy.pdf: 12 chunks
2026-03-02 20:31:07.209 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:227 -   ART_logylight_incomplete_datasheet.pdf: 6 chunks
2026-03-02 20:31:07.302 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:227 -   ART_product_catalog.pdf: 7 chunks
2026-03-02 20:31:07.307 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:227 -   AR

In [3]:
text_chunks = [c for c in all_chunks if c.mime_type.startswith("text/")]
# limit content size to 8k tokens for OpenAI embedding model
text_chunks = [c for c in text_chunks if len(c.content) < 8 * 1024]
await build_vector_store(
    text_chunks,
    OpenAIEmbeddings(model_name=EMBEDDING_MODEL),
    db_path=TEXT_VS_PATH,
    reset=True,
)

2026-03-02 20:33:26.859 | DEBUG    | conversational_toolkit.embeddings.openai:__init__:20 - OpenAI embeddings model loaded: text-embedding-3-small
2026-03-02 20:33:26.960 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:267 - Reset vector store collection at ../src/sme_kt_zh_collaboration_rag/db/vs_text
2026-03-02 20:33:26.960 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:275 - Embedding 333 chunks with <conversational_toolkit.embeddings.openai.OpenAIEmbeddings object at 0x17ddcf470> ...
2026-03-02 20:33:29.810 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (10, 1024)
2026-03-02 20:33:29.831 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:290 - Processed batch 1/34
2026-03-02 20:33:32.163 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (10, 1024)
2026-03-02 20:33:32.172 | INFO     | 

In [4]:
image_chunks = [c for c in all_chunks if c.mime_type.startswith("image/")]
await build_vector_store(
    image_chunks,
    Qwen3VLEmbeddings(),
    db_path=IMAGE_VS_PATH,
    reset=True,
    batch_size=1,
)

2026-03-02 20:34:15.832 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:267 - Reset vector store collection at ../src/sme_kt_zh_collaboration_rag/db/vs_image
2026-03-02 20:34:15.833 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:275 - Embedding 1151 chunks with <conversational_toolkit.embeddings.qwen_vl.Qwen3VLEmbeddings object at 0x17ddcd160> ...
2026-03-02 20:34:18.272 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:290 - Processed batch 1/1151
2026-03-02 20:34:18.872 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:290 - Processed batch 2/1151
2026-03-02 20:34:19.340 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:290 - Processed batch 3/1151
2026-03-02 20:34:19.641 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:290 - Processed batch 4/1151
2026-03-02 20:34:53.525 | INFO     | sme_

RuntimeError: MPS backend out of memory (MPS allocated: 11.24 GiB, other allocations: 6.23 GiB, max allowed: 20.13 GiB). Tried to allocate 3.01 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).